In [3]:
from huggingface_hub import login

# Run `huggingface-cli login` once in your terminal beforehand -- it caches the token
# permanently to disk, so login() below picks it up with no arguments needed.
try:
    login()
    print("✓ Hugging Face authentication successful!")
except Exception as e:
    print(f"⚠ Not logged in yet: {e}")
    print("Run this once in your terminal (outside the notebook): huggingface-cli login")

✓ Hugging Face authentication successful!


In [2]:
from datasets import load_dataset

ds = load_dataset("hassanjbara/BASEPROD", split="train", streaming=True)
print(ds)

IterableDataset({
    features: ['color', 'depth', 'depth_16bit', 'thermal', 'thermal_rgb'],
    num_shards: 106
})


In [ ]:
#Print the available features by inspecting the schema of the dataset so preprocessing adapts to whatever columns actually exist.
print("Available features:")
for name, feature in ds.features.items():
    print(f"  {name}: {feature}")

Available features:
  color: Image(mode=None, decode=True)
  depth: Image(mode=None, decode=True)
  depth_16bit: Image(mode=None, decode=True)
  thermal: Image(mode=None, decode=True)
  thermal_rgb: Image(mode=None, decode=True)


In [1]:
import numpy as np
from PIL import Image


def to_array(image, mode: str | None = None) -> np.ndarray:
    """Convert a PIL image (or already-decoded array) to a numpy array."""
    if isinstance(image, np.ndarray):
        return image
    if mode is not None:
        image = image.convert(mode)
    return np.array(image)


def preprocess_color(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize an RGB color image and scale pixel values to [0, 1]."""
    arr = to_array(image, mode="RGB")
    resized = np.array(Image.fromarray(arr).resize(size, Image.BILINEAR))
    return resized.astype(np.float32) / 255.0



def preprocess_depth(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a depth map and normalize to [0, 1] using its own min/max."""
    arr = to_array(image).astype(np.float32)
    resized = np.array(Image.fromarray(arr).resize(size, Image.NEAREST))
    depth_range = resized.max() - resized.min()
    if depth_range == 0:
        return np.zeros_like(resized)
    return (resized - resized.min()) / depth_range

def preprocess_thermal(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a thermal image and scale to [0, 1]."""
    arr = to_array(image).astype(np.float32)
    clipped = arr.astype(np.uint8) if arr.max() <= 255 else arr
    resized = np.array(Image.fromarray(clipped).resize(size, Image.BILINEAR))
    therm_range = resized.max() - resized.min()
    if therm_range == 0:
        return np.zeros_like(resized, dtype=np.float32)
    return (resized.astype(np.float32) - resized.min()) / therm_range

In [2]:
def preprocess_thermal(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a thermal image and scale to [0, 1]."""
    arr = to_array(image).astype(np.float32)
    clipped = arr.astype(np.uint8) if arr.max() <= 255 else arr
    resized = np.array(Image.fromarray(clipped).resize(size, Image.BILINEAR))
    therm_range = resized.max() - resized.min()
    if therm_range == 0:
        return np.zeros_like(resized, dtype=np.float32)
    return (resized.astype(np.float32) - resized.min()) / therm_range

def extract_depth_features(depth_arr: np.ndarray) -> dict:
    """Depth statistics: central tendency, spread, and coverage of valid pixels."""
    valid = depth_arr[depth_arr > 0]
    return {
        "depth_mean": float(valid.mean()) if valid.size else 0.0,
        "depth_std": float(valid.std()) if valid.size else 0.0,
        "depth_valid_ratio": float(valid.size / depth_arr.size),
    }

def extract_thermal_features(thermal_arr: np.ndarray) -> dict:
    """Thermal statistics: overall intensity spread as a proxy for hotspot presence."""
    return {
        "thermal_mean": float(thermal_arr.mean()),
        "thermal_std": float(thermal_arr.std()),
        "thermal_max": float(thermal_arr.max()),
    }

def extract_features(sample: dict) -> dict:
    """Run the full preprocessing + feature extraction pipeline on one sample."""
    features = {}
    if sample.get("color") is not None:
        features.update(extract_color_features(preprocess_color(sample["color"])))
    if sample.get("depth") is not None:
        features.update(extract_depth_features(preprocess_depth(sample["depth"])))
    if sample.get("thermal") is not None:
        features.update(extract_thermal_features(preprocess_thermal(sample["thermal"])))
    return features

In [3]:
import pandas as pd
from tqdm import tqdm

N_SAMPLES = 50  # adjust as needed; the dataset is streamed, so this pulls N_SAMPLES on demand

rows = [extract_features(sample) for sample in tqdm(ds.take(N_SAMPLES), total=N_SAMPLES)]
features_df = pd.DataFrame(rows)
print(f"Extracted features for {len(features_df)} samples")
features_df.head()

NameError: name 'ds' is not defined